# Demo 1 - Concurrency and workload isolation

**Why we are doing this.** Kenny's number one complaint: *"large daily file loads, sync processes, and reporting stored procedures experience significant slowdowns when running simultaneously."* In Synapse all of these jobs share one fixed pool of compute, so Power BI users wait behind the loads.

**What we will prove.** We run the same three jobs, with the same 80 Power BI users, two ways:
1. **Act 1 (Synapse):** all on one shared warehouse (imitates their Synapse today). Dashboard queries wait in a queue.
2. **Act 2 (Snowflake):** each job on its own warehouse, and the Power BI warehouse automatically adds clusters when many users arrive at once. The jobs stop competing, and the dashboards stop queueing.

Every warehouse is the same size (Medium), so the difference comes from how the compute is arranged, not from a bigger engine. Then a summary table and a query show, in numbers, time spent waiting versus time spent working.

**How to read this notebook.** Every cell has one of four labels:
- **SETUP - ALREADY DONE (read only):** code that already exists in the account. It is shown so the engineers can see how it was built. You do not run it.
- **RUN DURING DEMO:** a SQL cell you run live. Click inside it and press the run button (or Cmd+Enter).
- **IF ASKED:** optional; run it only if the audience asks about that topic. Skip it in the 45-minute cut.
- **SAY THIS:** your talk track for that moment.

> **SAY THIS:** Same data, same jobs. We just stop making them fight over one engine.

## The three jobs (already built)
These three stored procedures stand in for Somatus's real workload. They are reused in both acts, which keeps the comparison fair: only the compute arrangement changes.

- **SP_DAILY_LOAD** imitates the large daily client file load. It inserts about 30 million claim rows.
- **SP_BUILD_REPORTING** imitates their heavy reporting stored procedure. It joins 100 million claims to 2 million members and aggregates them.
- **SP_BI_BURST(80)** imitates 80 Power BI users running dashboard queries at the same time.

### SETUP - ALREADY DONE (read only, do not run)
**Stored procedure: the daily load**

```sql
CREATE OR REPLACE PROCEDURE SOMATUS_DEMO.OPS.SP_DAILY_LOAD() RETURNS VARCHAR LANGUAGE SQL AS
BEGIN
  INSERT INTO SOMATUS_DEMO.BRONZE.CLAIMS_DAILY_LOAD SELECT * FROM SOMATUS_DEMO.SILVER.CLAIMS SAMPLE ROW (30);
  RETURN 'daily load done';
END;
```

### SETUP - ALREADY DONE (read only, do not run)
**Stored procedure: the heavy reporting job**

```sql
CREATE OR REPLACE PROCEDURE SOMATUS_DEMO.OPS.SP_BUILD_REPORTING() RETURNS VARCHAR LANGUAGE SQL AS
BEGIN
  CREATE OR REPLACE TABLE SOMATUS_DEMO.GOLD.RPT_MEMBER_COST AS
  SELECT m.CLIENT_ID, m.CKD_STAGE, m.STATE, DATE_TRUNC('month', c.SERVICE_DATE) AS SVC_MONTH,
         COUNT(DISTINCT c.MEMBER_ID) AS MEMBERS, COUNT(*) AS CLAIMS, SUM(c.PAID_AMOUNT) AS TOTAL_PAID,
         MEDIAN(c.PAID_AMOUNT) AS MEDIAN_PAID
  FROM SOMATUS_DEMO.SILVER.CLAIMS c JOIN SOMATUS_DEMO.SILVER.MEMBERS m ON c.MEMBER_ID = m.MEMBER_ID
  GROUP BY 1,2,3,4;
  RETURN 'reporting SP done';
END;
```

### SETUP - ALREADY DONE (read only, do not run)
**Tasks: one wrapper per job and per mode. The only difference between modes is the warehouse.**

```sql
-- Act 1, shared mode: all three point at the same warehouse
CREATE TASK T_SHARED_LOAD      WAREHOUSE=WH_SHARED_SYNAPSE_SIM QUERY_TAG='workload=ingest;mode=shared' AS CALL SP_DAILY_LOAD();
CREATE TASK T_SHARED_TRANSFORM WAREHOUSE=WH_SHARED_SYNAPSE_SIM ... AS CALL SP_BUILD_REPORTING();
CREATE TASK T_SHARED_BI        WAREHOUSE=WH_SHARED_SYNAPSE_SIM ... AS CALL SP_BI_BURST(80);
-- Act 2, isolated mode: one warehouse per workload
CREATE TASK T_ISO_LOAD      WAREHOUSE=WH_INGEST    ... AS CALL SP_DAILY_LOAD();
CREATE TASK T_ISO_TRANSFORM WAREHOUSE=WH_TRANSFORM ... AS CALL SP_BUILD_REPORTING();
CREATE TASK T_ISO_BI        WAREHOUSE=WH_BI        ... AS CALL SP_BI_BURST(80);
-- All four warehouses are MEDIUM.
-- WH_BI is multi-cluster (auto-scale): MIN_CLUSTER_COUNT=1, MAX_CLUSTER_COUNT=4
```

### RUN DURING DEMO
Set the role and database for this notebook. Run this first, every time you open the notebook.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE SCHEMA OPS;

### RUN DURING DEMO
Show the size of the data: about 100 million claims and 2 million members. This tells the audience the test is not a toy.

In [ ]:
%%sql -r data_size
SELECT (SELECT COUNT(*) FROM SOMATUS_DEMO.SILVER.CLAIMS)  AS CLAIMS,
       (SELECT COUNT(*) FROM SOMATUS_DEMO.SILVER.MEMBERS) AS MEMBERS;

### RUN DURING DEMO
Show the warehouses. All four are the same size (Medium). Point out WH_SHARED_SYNAPSE_SIM (one engine shared by everything, like their Synapse pool) and WH_INGEST, WH_TRANSFORM and WH_BI (one engine per workload). WH_BI can grow from 1 to 4 clusters.

In [ ]:
%%sql -r warehouses
SHOW WAREHOUSES LIKE 'WH_%'
->> SELECT "name" AS WAREHOUSE, "size" AS SIZE,
           "min_cluster_count" AS MIN_CLUSTERS, "max_cluster_count" AS MAX_CLUSTERS,
           "auto_suspend" AS AUTO_SUSPEND_SECS, "comment" AS PURPOSE
    FROM $1
    WHERE "name" IN ('WH_SHARED_SYNAPSE_SIM','WH_INGEST','WH_TRANSFORM','WH_BI')
    ORDER BY IFF("name" = 'WH_SHARED_SYNAPSE_SIM', 0, 1), "name";

## Act 1 - "Your Synapse today": everything on one shared warehouse
#### WH_SHARED_SYNAPSE_SIM


### RUN DURING DEMO
Start all three jobs at the same moment on the single shared warehouse: the daily load, the reporting procedure, and 80 Power BI users. The commands return immediately because the jobs run in the background.

In [ ]:
%%sql -r act1
EXECUTE TASK T_SHARED_LOAD;
EXECUTE TASK T_SHARED_TRANSFORM;
EXECUTE TASK T_SHARED_BI;

## Act 2 - Each job on its own warehouse, and Power BI scales out automatically
### Workload isolation in Snowflake, using a warehouse per workload: WH_INGEST, WH_TRANSFORM, WH_BI

### RUN DURING DEMO
Run the same three jobs again, with the same 80 Power BI users, but each job on its own Medium warehouse. WH_BI is allowed to grow to 4 clusters: it starts with 1 and Snowflake adds more only if dashboard queries start to queue.

In [ ]:
%%sql -r act2
ALTER WAREHOUSE WH_BI SET MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 4;   -- auto scale-out for Power BI
EXECUTE TASK T_ISO_LOAD;
EXECUTE TASK T_ISO_TRANSFORM;
EXECUTE TASK T_ISO_BI;

### RUN DURING DEMO
Run this once Act 2 has finished (about 30 seconds after the Act 2 cell). Each row is one cluster of WH_BI and how many of the 80 dashboard queries it served. More than one row means Snowflake added clusters by itself. WH_BI is allowed up to 4 clusters; it adds only what the peak needs, and you pay only for the clusters that run.

In [ ]:
%%sql -r cluster_proof
-- Only the most recent Act 2 Power BI run is counted.
WITH last_run AS (
  SELECT MAX(QUERY_START_TIME) AS RUN_START
  FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.TASK_HISTORY(
         SCHEDULED_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP()), TASK_NAME => 'T_ISO_BI'))
)
SELECT q.CLUSTER_NUMBER, COUNT(*) AS DASHBOARD_QUERIES,
       ROUND(SUM(q.QUEUED_OVERLOAD_TIME)/1000,1) AS SECONDS_SPENT_WAITING
FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.QUERY_HISTORY_BY_WAREHOUSE(
       WAREHOUSE_NAME => 'WH_BI', END_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP()), RESULT_LIMIT => 1000)) q
JOIN last_run r ON q.START_TIME >= r.RUN_START
WHERE q.QUERY_TYPE = 'INSERT' AND q.QUERY_TAG = 'workload=bi;mode=isolated'
GROUP BY 1 ORDER BY 1;

> **SAY THIS:** Same data, same jobs, same size of engine. The load can no longer take compute away from the dashboards, because they run on separate engines that all read the same single copy of data. And when many users arrive at once, Snowflake adds clusters automatically and removes them when the peak passes. You only pay while they run.

## Proof - waiting versus working, in numbers

### RUN DURING DEMO
The one-table summary: time to finish each job, Act 1 versus Act 2, from the most recent run of each act. Run Act 1 and Act 2 as many times as you like; this always shows the latest pair. The detailed waiting-versus-working breakdown follows below if the engineers want it.

In [ ]:
%%sql -r summary
-- Most recent successful run of each task, pivoted into Act 1 vs Act 2.
WITH latest AS (
  SELECT NAME,
         DATEDIFF('millisecond', QUERY_START_TIME, COMPLETED_TIME)/1000 AS SECS
  FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.TASK_HISTORY(
         SCHEDULED_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP())))
  WHERE NAME IN ('T_SHARED_LOAD','T_SHARED_TRANSFORM','T_SHARED_BI','T_ISO_LOAD','T_ISO_TRANSFORM','T_ISO_BI')
    AND STATE = 'SUCCEEDED'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY NAME ORDER BY COMPLETED_TIME DESC) = 1
), jobs AS (
  SELECT CASE WHEN NAME LIKE '%LOAD' THEN 1 WHEN NAME LIKE '%TRANSFORM' THEN 2 ELSE 3 END AS SORT_ORDER,
         CASE WHEN NAME LIKE '%LOAD' THEN 'Daily load'
              WHEN NAME LIKE '%TRANSFORM' THEN 'Reporting procedure'
              ELSE 'Power BI burst (80 users)' END AS JOB,
         IFF(NAME LIKE 'T_SHARED%', SECS, NULL) AS ACT1_S,
         IFF(NAME LIKE 'T_ISO%',    SECS, NULL) AS ACT2_S
  FROM latest
), pivoted AS (
  SELECT SORT_ORDER, JOB, MAX(ACT1_S) AS A1, MAX(ACT2_S) AS A2 FROM jobs GROUP BY 1, 2
)
SELECT JOB,
       ROUND(A1,1) || 's' AS "Act 1: shared (Synapse)",
       ROUND(A2,1) || 's (' ||
         IFF(A2 <= A1, ROUND(A1/NULLIF(A2,0),1) || 'x faster', ROUND(A2/NULLIF(A1,0),1) || 'x slower')
         || ')' AS "Act 2: isolated + scale-out (Snowflake)"
FROM pivoted
ORDER BY SORT_ORDER;

> **SAY THIS:** Same three jobs, same code, same data, same size of engine. Sharing one engine, every job took longer, because they waited for each other. Separated, and with Power BI scaling out, every job finished faster. *(Read the speed-ups straight off the table.)*

### RUN DURING DEMO
Run this once both acts have finished. It counts only the most recent run of each act, so earlier rehearsals never mix in. Read the Power BI rows: in Act 1 each dashboard query waited about 17 seconds for under 2 seconds of work; in Act 2 the wait drops to a few seconds, and CLUSTERS_USED shows WH_BI scaled out. The load and the reporting job show zero wait in Act 2.

In [ ]:
%%sql -r proof
-- Every job carries a query tag such as 'workload=bi;mode=shared', so each act gets its own rows.
-- Only the most recent run of each act is counted, so earlier rehearsals never mix in.
WITH latest_task AS (
  SELECT NAME, QUERY_START_TIME
  FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.TASK_HISTORY(
         SCHEDULED_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP())))
  WHERE NAME IN ('T_SHARED_LOAD','T_SHARED_TRANSFORM','T_SHARED_BI','T_ISO_LOAD','T_ISO_TRANSFORM','T_ISO_BI')
    AND STATE = 'SUCCEEDED'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY NAME ORDER BY QUERY_START_TIME DESC) = 1
), last_run AS (
  SELECT IFF(NAME LIKE 'T_SHARED%', 'shared', 'isolated') AS MODE, MIN(QUERY_START_TIME) AS RUN_START
  FROM latest_task GROUP BY 1
)
SELECT CASE q.MODE
         WHEN 'shared'   THEN 'Act 1 - One shared warehouse (Synapse today)'
         WHEN 'isolated' THEN 'Act 2 - One warehouse per job + BI scale-out' END AS ACT,
       CASE SPLIT_PART(SPLIT_PART(q.QUERY_TAG,';',1),'=',2)
         WHEN 'ingest' THEN 'Daily load' WHEN 'transform' THEN 'Reporting procedure'
         ELSE 'Power BI dashboard query' END AS JOB,
       q.WAREHOUSE_NAME,
       COUNT(*)                                  AS QUERIES,
       ROUND(AVG(q.QUEUED_OVERLOAD_TIME)/1000,1) AS AVG_SECONDS_WAITING,
       ROUND(AVG(q.EXECUTION_TIME)/1000,1)       AS AVG_SECONDS_WORKING,
       ROUND(AVG(q.TOTAL_ELAPSED_TIME)/1000,1)   AS AVG_SECONDS_TOTAL,
       COUNT(DISTINCT q.CLUSTER_NUMBER)          AS CLUSTERS_USED
FROM (SELECT *, SPLIT_PART(QUERY_TAG,'mode=',2) AS MODE
      FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.QUERY_HISTORY(
             END_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP()), RESULT_LIMIT => 10000))
      WHERE QUERY_TAG LIKE 'workload=%;mode=%'
        AND QUERY_TYPE IN ('INSERT','CREATE_TABLE_AS_SELECT')) q
JOIN last_run r ON r.MODE = q.MODE AND q.START_TIME >= r.RUN_START
GROUP BY 1,2,3 ORDER BY 1,2;

> **SAY THIS:** This is not a claim, it is the query history from the run you just watched.

## What we just proved

| # | Your challenge today | What you saw |
|---|---|---|
| 1 | Loads, reporting and Power BI slow each other down when they run together | On one shared warehouse (your Synapse today), every job ran **2.5 to 4 times slower** than on its own warehouse |
| 2 | Power BI users wait behind the loads | Each dashboard query **waited about 17 seconds for under 2 seconds of work**. With its own warehouse, the wait dropped to **about 2 to 3 seconds** |
| 3 | A big daily load holds up everything else | The daily load finished in **about 7 seconds instead of 18**, and the reporting job in **about 8 seconds instead of 22** |
| 4 | Monday-morning and month-end peaks | 80 dashboard users at once: WH_BI **added clusters by itself (3 of the 4 allowed)** and removed them after the peak |
| 5 | "Is it just a bigger engine?" | No. **Every warehouse was the same size (Medium).** Only the arrangement changed |
| 6 | Copying data to give teams their own compute | **No copies.** All warehouses read the same single copy of the tables |
| 7 | Paying for capacity you do not use | Warehouses **suspend when idle** and extra clusters exist only during the peak. You pay per second they run |

*Numbers are from the latest rehearsal; read the summary table above for today's run.*

**Next:** the data is loaded and the jobs no longer fight. Now let's build Silver and Gold so only new data is processed.

## Bonus 1 (IF ASKED) - Scale up for one heavy job, then back down

### IF ASKED - skip in the 45-minute cut
Use this if someone asks "what if one job is just heavy?" Resize the transform warehouse to extra large for one heavy job, then back to medium. The resize takes seconds and you pay for extra large only while the job runs.

In [ ]:
%%sql -r scale_up
USE WAREHOUSE WH_TRANSFORM;                              -- run the job on the transform warehouse
ALTER WAREHOUSE WH_TRANSFORM SET WAREHOUSE_SIZE = 'XLARGE';
CALL SP_BUILD_REPORTING();
ALTER WAREHOUSE WH_TRANSFORM SET WAREHOUSE_SIZE = 'MEDIUM';  -- back to the demo size
USE WAREHOUSE COMPUTE_WH;

## Bonus 2 (IF ASKED) - Readers are never blocked by writers

### IF ASKED - skip in the 45-minute cut
Use this if someone asks about locking. Run it while a load is inserting into the same table (start T_ISO_LOAD again first if needed). It returns instantly from a consistent snapshot. There is no table locking.

In [ ]:
%%sql -r no_lock
SELECT COUNT(*) FROM SOMATUS_DEMO.BRONZE.CLAIMS_DAILY_LOAD;